# 01 — Exploratory Data Analysis & Data Prep (Colab edition)

**MIS 3080 Final Project — Movie Recommender**

This notebook does two things in one pass:

1. **Explores** the raw TMDB 5000 movies and credits CSVs (shapes, types, missingness, distributions, relationships).
2. **Engineers features** and produces `movies_clean.parquet` — the single cleaned dataset that the rest of the project (rating regressor, embeddings, Streamlit app) consumes.

### Running this notebook in Google Colab

1. Click **Runtime → Run all**.
2. When the upload cell runs, upload **both** files: `tmdb_5000_movies.csv` and `tmdb_5000_credits.csv`.
3. The final cell will auto-download the cleaned `movies_clean.parquet` to your computer. Keep that file — the next two notebooks (model training, embeddings) consume it.

### Data sources

- `tmdb_5000_movies.csv` — 4,803 movies with budget, revenue, genres, plot overview, etc.
- `tmdb_5000_credits.csv` — cast and crew per movie, joinable on `movie_id`.

Both come from the public TMDB 5000 dataset on Kaggle.


## 0. Setup — Colab detection & dependencies


In [ ]:
# Colab pre-installs pandas, numpy, scikit-learn, matplotlib, seaborn.
# We just need pyarrow for parquet I/O. Quiet install — no-op if already present.
%pip install -q pyarrow


In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("running in Colab :" , IN_COLAB)


### Upload the two CSVs (Colab only)

When the next cell runs in Colab, a file-picker will appear. **Select both CSVs in the same dialog** (Ctrl/Cmd-click).

Outside Colab (e.g. running locally), this cell is a no-op and the notebook will look for the files in `../data/` next to the project folder.


In [ ]:
from pathlib import Path

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()  # multi-select supported
    DATA_DIR = Path("/content")
    for name in uploaded:
        print(f"uploaded {name} ({len(uploaded[name])/1024:.1f} KB)")
else:
    # Local fallback: notebook lives in notebooks/, data in ../data/
    DATA_DIR = Path.cwd().parent / "data" if Path.cwd().name == "notebooks" else Path.cwd() / "data"

ARTIFACTS_DIR = Path("/content") if IN_COLAB else (DATA_DIR.parent / "artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR     :", DATA_DIR)
print("ARTIFACTS_DIR:", ARTIFACTS_DIR)

# Sanity check — both files must be in DATA_DIR
expected = ["tmdb_5000_movies.csv", "tmdb_5000_credits.csv"]
missing = [f for f in expected if not (DATA_DIR / f).exists()]
assert not missing, f"missing files in {DATA_DIR}: {missing}"
print("both CSVs found OK")


In [ ]:
import ast
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")


## 1. Load raw data


In [ ]:
movies = pd.read_csv(DATA_DIR / "tmdb_5000_movies.csv")
credits = pd.read_csv(DATA_DIR / "tmdb_5000_credits.csv")
print("movies :", movies.shape)
print("credits:", credits.shape)


In [ ]:
movies.head(2)


In [ ]:
credits.head(2)


### What the columns look like
Note in particular: `genres`, `keywords`, `production_companies`, `production_countries`, `spoken_languages` (in `movies`) and `cast`, `crew` (in `credits`) are stored as **stringified JSON** — they need to be parsed before they're useful.


In [ ]:
movies.dtypes


In [ ]:
# Missing values in raw data
miss = pd.DataFrame({
    "missing": movies.isna().sum(),
    "pct": (movies.isna().mean() * 100).round(2),
}).sort_values("missing", ascending=False)
miss[miss["missing"] > 0]


## 2. Inspect raw distributions


### Target variable: `vote_average`
This is what the ML rating regressor will predict. We want to see its shape and watch out for `0`s, which probably mean *no votes / unknown* rather than a real rating of zero.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
movies["vote_average"].plot.hist(bins=40, ax=ax, edgecolor="white")
ax.set_title("Distribution of vote_average (raw)")
ax.set_xlabel("vote_average")
plt.show()
movies["vote_average"].describe()


In [ ]:
# How many movies have 0 votes? Their rating is meaningless.
(movies["vote_count"] == 0).sum(), (movies["vote_average"] == 0).sum()


### Vote count is heavily right-skewed
A handful of blockbusters have tens of thousands of votes; many obscure titles have <50. We'll need to **filter low-vote rows for training** so the regressor learns from reliable labels — but **keep them in the embedding index** so users can still discover them.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
movies["vote_count"].plot.hist(bins=60, ax=axes[0], edgecolor="white")
axes[0].set_title("vote_count (raw)")
np.log1p(movies["vote_count"]).plot.hist(bins=60, ax=axes[1], edgecolor="white")
axes[1].set_title("log1p(vote_count)")
plt.tight_layout(); plt.show()


### `budget` and `revenue` — zeros mean missing

In [ ]:
zero_budget = (movies["budget"] == 0).sum()
zero_revenue = (movies["revenue"] == 0).sum()
print(f"budget == 0 : {zero_budget} rows ({zero_budget/len(movies):.1%})")
print(f"revenue == 0: {zero_revenue} rows ({zero_revenue/len(movies):.1%})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
np.log1p(movies.loc[movies["budget"]>0, "budget"]).plot.hist(bins=50, ax=axes[0], edgecolor="white")
axes[0].set_title("log1p(budget) - non-zero only")
np.log1p(movies.loc[movies["revenue"]>0, "revenue"]).plot.hist(bins=50, ax=axes[1], edgecolor="white")
axes[1].set_title("log1p(revenue) - non-zero only")
plt.tight_layout(); plt.show()


### Release year distribution

In [ ]:
years = pd.to_datetime(movies["release_date"], errors="coerce").dt.year
fig, ax = plt.subplots(figsize=(10, 3.5))
years.dropna().astype(int).plot.hist(bins=range(1915, 2020), ax=ax, edgecolor="white")
ax.set_title("Movies by release year"); ax.set_xlabel("year")
plt.show()
print("year range:", years.min(), "to", years.max())


### Top genres (quick parse)

In [ ]:
# Quick inline parse just for visualization - formal parsing is in section 4
import collections
counter = collections.Counter()
for raw in movies["genres"].dropna():
    for g in ast.literal_eval(raw):
        counter[g["name"]] += 1
top_genres = pd.Series(counter).sort_values(ascending=True).tail(15)
fig, ax = plt.subplots(figsize=(8, 5))
top_genres.plot.barh(ax=ax)
ax.set_title("Top genres by movie count"); ax.set_xlabel("count")
plt.show()


## 3. Findings → data-prep decisions

From the EDA above:

| Finding | Decision |
|---|---|
| `genres`, `keywords`, `cast`, `crew`, etc. are JSON-as-string | Parse into Python lists, then extract names |
| ~22% of `budget` and ~30% of `revenue` are `0` (means missing) | Replace `0 → NaN` so they're not treated as real zeros |
| `budget`, `revenue`, `popularity`, `vote_count` are heavily right-skewed | Add `log1p`-transformed copies for downstream use |
| Many movies have `vote_count < 50` → noisy `vote_average` | Add a `train_eligible` flag (≥50 votes) to filter the regressor's training set, but **keep all rows** for the embedding lookup |
| Plot text + tagline + genres + keywords are the user's natural query targets | Concatenate into a single `embed_text` field |
| Need to join cast/crew | Merge `credits` on `movies.id == credits.movie_id` |

The remaining sections execute these decisions.


## 4. Parse JSON-string columns


In [ ]:
def parse_json_col(series: pd.Series) -> pd.Series:
    """Parse a column of stringified JSON/Python-literal lists into real lists.

    Falls back gracefully on malformed values, returning [].
    Uses ast.literal_eval first (TMDB uses single-quoted Python repr)
    then json.loads as a backup.
    """
    def _parse(x):
        if pd.isna(x) or x == "":
            return []
        try:
            return ast.literal_eval(x)
        except (ValueError, SyntaxError):
            try:
                return json.loads(x)
            except Exception:
                return []
    return series.apply(_parse)

for col in ["genres", "keywords", "production_companies",
            "production_countries", "spoken_languages"]:
    movies[col] = parse_json_col(movies[col])

credits["cast"] = parse_json_col(credits["cast"])
credits["crew"] = parse_json_col(credits["crew"])

movies["genres"].iloc[0]


## 5. Merge movies + credits


In [ ]:
credits_renamed = credits.rename(columns={"movie_id": "id"}).drop(columns=["title"])
df = movies.merge(credits_renamed, on="id", how="left")
print("merged shape:", df.shape)
print("merge sanity - rows missing cast info:", df["cast"].isna().sum())


## 6. Feature engineering


### Genres — list of names + primary genre

In [ ]:
df["genre_names"] = df["genres"].apply(lambda lst: [g["name"] for g in lst])
df["primary_genre"] = df["genre_names"].apply(lambda lst: lst[0] if lst else "Unknown")
df["n_genres"] = df["genre_names"].apply(len)
df[["title", "genre_names", "primary_genre", "n_genres"]].head()


### Keywords — list of names

In [ ]:
df["keyword_names"] = df["keywords"].apply(lambda lst: [k["name"] for k in lst])
df["n_keywords"] = df["keyword_names"].apply(len)
df[["title", "keyword_names", "n_keywords"]].head()


### Release date → year

In [ ]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df[["title", "release_date", "release_year"]].head()


### Director (from crew) and top-3 cast

In [ ]:
def get_director(crew_list):
    for member in crew_list:
        if member.get("job") == "Director":
            return member.get("name")
    return None

df["director"] = df["crew"].apply(get_director)
df["top_cast"] = df["cast"].apply(lambda lst: [m["name"] for m in lst[:3]])
df["lead_actor"] = df["top_cast"].apply(lambda lst: lst[0] if lst else None)
df["cast_size"] = df["cast"].apply(len)

print("rows with director :", df["director"].notna().sum())
print("rows with lead_actor:", df["lead_actor"].notna().sum())
df[["title", "director", "top_cast", "cast_size"]].head()


### Replace `0` with `NaN` for budget / revenue, then log-transform skewed numerics

In [ ]:
df["budget"]  = df["budget"].replace(0, np.nan)
df["revenue"] = df["revenue"].replace(0, np.nan)

df["log_budget"]      = np.log1p(df["budget"])
df["log_revenue"]     = np.log1p(df["revenue"])
df["log_vote_count"]  = np.log1p(df["vote_count"])
df["log_popularity"]  = np.log1p(df["popularity"])

df[["budget", "log_budget", "revenue", "log_revenue",
    "vote_count", "log_vote_count"]].describe().round(2)


### Build `embed_text` — the field the AI component will embed

We concatenate the fields a user is likely to type a query against: the **title**, **tagline**, **plot overview**, **genre names**, and **keyword names**. A delimiter (` | `) separates them so the embedding model sees clear field boundaries.


In [ ]:
def build_embed_text(row):
    parts = [
        str(row.get("title", "") or ""),
        str(row.get("tagline", "") or ""),
        str(row.get("overview", "") or ""),
        " ".join(row["genre_names"]),
        " ".join(row["keyword_names"]),
    ]
    return " | ".join(p for p in parts if p)

df["embed_text"] = df.apply(build_embed_text, axis=1)

for i in [0, 100, 500]:
    print(f"--- row {i}: {df['title'].iloc[i]} ---")
    print(df["embed_text"].iloc[i][:280])
    print()


### `train_eligible` flag

For training the rating regressor we only want movies with **enough votes for the rating to be reliable** AND **a known year and rating**. This is a *flag*, not a filter — the row is still kept in the dataset so the embedding lookup can return it.


In [ ]:
df["train_eligible"] = (
    (df["vote_count"].fillna(0) >= 50)
    & df["vote_average"].notna()
    & (df["vote_average"] > 0)
    & df["release_year"].notna()
)
df["train_eligible"].value_counts()


## 7. Final inspection — sanity checks before saving


In [ ]:
sample = df.sample(min(2000, len(df)), random_state=0)
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(sample["log_vote_count"], sample["vote_average"], alpha=0.3, s=10)
ax.axvline(np.log1p(50), color="red", linestyle="--", label="vote_count = 50")
ax.set_xlabel("log1p(vote_count)"); ax.set_ylabel("vote_average")
ax.set_title("Rating vs. vote count - low-vote rows are noisy")
ax.legend(); plt.show()


In [ ]:
eligible = df[df["train_eligible"]]
genre_stats = (
    eligible.groupby("primary_genre")["vote_average"]
    .agg(["mean", "count"])
    .query("count >= 30")
    .sort_values("mean", ascending=True)
)
fig, ax = plt.subplots(figsize=(8, 6))
genre_stats["mean"].plot.barh(ax=ax)
ax.set_title("Mean vote_average by primary genre (>=30 eligible movies)")
ax.set_xlabel("mean vote_average")
plt.show()
genre_stats


In [ ]:
plot_df = eligible.dropna(subset=["log_budget"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(plot_df["log_budget"], plot_df["vote_average"], alpha=0.25, s=10)
ax.set_xlabel("log1p(budget)"); ax.set_ylabel("vote_average")
ax.set_title("Budget vs rating (training-eligible rows)")
plt.show()
plot_df[["log_budget", "vote_average"]].corr().round(3)


## 8. Save the cleaned dataset


In [ ]:
keep_cols = [
    # identity / display
    "id", "title", "original_title", "original_language",
    "release_date", "release_year",
    # numeric features
    "budget", "revenue", "popularity", "runtime", "vote_average", "vote_count",
    "log_budget", "log_revenue", "log_popularity", "log_vote_count",
    # text features
    "tagline", "overview", "homepage", "status",
    # categorical / list features
    "genre_names", "primary_genre", "n_genres",
    "keyword_names", "n_keywords",
    "director", "top_cast", "lead_actor", "cast_size",
    # derived combined fields
    "embed_text",
    # flags
    "train_eligible",
]
clean = df[keep_cols].copy()

out_path = ARTIFACTS_DIR / "movies_clean.parquet"
clean.to_parquet(out_path, index=False)

print(f"Wrote {out_path}")
print(f"  rows           : {len(clean):,}")
print(f"  columns        : {len(clean.columns)}")
print(f"  size           : {out_path.stat().st_size/1024:.1f} KB")
print(f"  train_eligible : {clean['train_eligible'].sum():,} rows")


In [ ]:
reloaded = pd.read_parquet(out_path)
assert reloaded.shape == clean.shape
reloaded.head()


### Download the cleaned parquet (Colab)

The next cell triggers a browser download of `movies_clean.parquet` so you can save it locally and feed it into the next two notebooks. Skipped automatically when running outside Colab.


In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(out_path))
else:
    print("Local run - file already saved at", out_path)


## 9. Next steps

With `movies_clean.parquet` in hand, the next two notebooks consume it:

- **`02_train_model.ipynb`** — fit the rating regressor (XGBoost vs. linear baseline) on the `train_eligible` subset. Saves `model.joblib` and a held-out evaluation report.
- **`03_build_embeddings.ipynb`** — encode every row's `embed_text` with `sentence-transformers/all-MiniLM-L6-v2` and save `embeddings.npy` for the Streamlit app to load at startup.

After both, `app.py` ties them together: query → embedding similarity (top 50 candidates) → ML score (re-rank) → top-N recommendations.
